In [ ]:
pip install pandas python-dotenv faiss-cpu langchain-openai openai


In [ ]:
import os
from dotenv import load_dotenv

import faiss
import numpy as np
import pandas as pd

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print("Model:", OPENAI_MODEL)
print("OpenAI key found:", bool(OPENAI_API_KEY))


In [ ]:
changes_path = "data/test_changerequests.csv"
incidents_path = "data/test_incidents.csv"

changes_df = pd.read_csv(changes_path)
incidents_df = pd.read_csv(incidents_path)

print("Changes shape:", changes_df.shape)
print("Incidents shape:", incidents_df.shape)

changes_df.head()


In [ ]:
incidents_df.head()


In [ ]:
texts = []

for _, row in changes_df.iterrows():
    texts.append(
        f"CHANGE | service={row['service']} | type={row['change_type']} | "
        f"description={row['description']} | risk={row['risk_level']}"
    )

for _, row in incidents_df.iterrows():
    texts.append(
        f"INCIDENT | summary={row['incident_summary']} | "
        f"root_cause={row['root_cause']} | resolution={row['resolution']}"
    )

print("Total records prepared:", len(texts))
texts[:5]


In [ ]:
# build vector store and FIASS for similarity serach

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

vectors = embeddings.embed_documents(texts)
vectors_np = np.array(vectors).astype("float32")

index = faiss.IndexFlatL2(vectors_np.shape[1])
index.add(vectors_np)

print("Vector store built successfully")
print("Embedding dimension:", vectors_np.shape[1])


In [ ]:
#retrieval

def retrieve_similar(change_request: str, top_k: int = 3):
    query_vector = embeddings.embed_query(change_request)
    query_np = np.array([query_vector]).astype("float32")
    distances, indices = index.search(query_np, top_k)
    return [texts[i] for i in indices[0] if i < len(texts)]


In [ ]:
# risk score
def calculate_risk(change_request: str):
    text = change_request.lower()
    score = 0

    if "database" in text or "schema" in text or "migration" in text:
        score += 3

    if "payment" in text or "auth" in text or "login" in text:
        score += 2

    if "deploy" in text or "upgrade" in text or "version" in text:
        score += 1

    if score >= 6:
        level = "High"
    elif score >= 3:
        level = "Medium"
    else:
        level = "Low"

    return {
        "risk_score": score,
        "risk_level": level,
    }


In [ ]:
# LLM analysis
def generate_analysis(change_request: str, similar_records: list, risk_result: dict):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL, temperature=0)

    prompt = f'''
You are an AI assistant for software change risk analysis.

User change request:
{change_request}

Similar past records:
{chr(10).join(similar_records)}

Risk level: {risk_result["risk_level"]}
Risk score: {risk_result["risk_score"]}

Provide:
1. impacted areas
2. possible failures
3. recommended tests
4. rollback checks

Keep the answer concise and practical.
'''

    response = llm.invoke(prompt)
    return response.content


In [ ]:
# testing flow with user input
change_request = "Deploy payment API with database schema change for transaction table"
change_request


In [ ]:
similar_records = retrieve_similar(change_request, top_k=3)
similar_records


In [ ]:
risk_result = calculate_risk(change_request)
risk_result


In [ ]:
analysis = generate_analysis(change_request, similar_records, risk_result)
print(analysis)


In [ ]:
result = {
    "change_request": change_request,
    "similar_records": similar_records,
    "risk_level": risk_result["risk_level"],
    "risk_score": risk_result["risk_score"],
    "analysis": analysis,
}

result
